In [1]:
pip install opencv-python numpy pillow tqdm

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import cv2
import random
import numpy as np
from tqdm import tqdm
from pathlib import Path

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
def load_grayscale(path):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)

    if img is None:
        raise ValueError(f"Cannot read {path}")

    return img


def save_output(img, mask, output_dir,
                label, class_name,
                original_name):

    img_dir = Path(output_dir) / "images" / label / class_name
    mask_dir = Path(output_dir) / "masks" / label / class_name

    img_dir.mkdir(parents=True, exist_ok=True)
    mask_dir.mkdir(parents=True, exist_ok=True)

    cv2.imwrite(str(img_dir / original_name), img)

    cv2.imwrite(str(mask_dir / original_name), mask)

In [4]:
def copy_move_forgery(img):

    h, w = img.shape

    patch_size = random.randint(32, 96)

    x1 = random.randint(0, w - patch_size)
    y1 = random.randint(0, h - patch_size)

    patch = img[
        y1:y1 + patch_size,
        x1:x1 + patch_size
    ].copy()

    x2 = random.randint(0, w - patch_size)
    y2 = random.randint(0, h - patch_size)

    forged = img.copy()

    forged[
        y2:y2 + patch_size,
        x2:x2 + patch_size
    ] = patch

    mask = np.zeros_like(img)

    mask[
        y2:y2 + patch_size,
        x2:x2 + patch_size
    ] = 255

    return forged, mask

In [5]:
def splice_forgery(img1, img2):

    h, w = img1.shape

    patch_size = random.randint(64, 128)

    x_src = random.randint(0, w - patch_size)
    y_src = random.randint(0, h - patch_size)

    patch = img2[
        y_src:y_src + patch_size,
        x_src:x_src + patch_size
    ]

    x_dst = random.randint(0, w - patch_size)
    y_dst = random.randint(0, h - patch_size)

    forged = img1.copy()

    forged[
        y_dst:y_dst + patch_size,
        x_dst:x_dst + patch_size
    ] = patch

    mask = np.zeros_like(img1)

    mask[
        y_dst:y_dst + patch_size,
        x_dst:x_dst + patch_size
    ] = 255

    return forged, mask

In [6]:
def synthetic_defect_injection(img):

    forged = img.copy()

    mask = np.zeros_like(img)

    num_defects = random.randint(5, 20)

    for _ in range(num_defects):

        radius = random.randint(5, 15)

        x = random.randint(radius,
                           img.shape[1] - radius)

        y = random.randint(radius,
                           img.shape[0] - radius)

        intensity = random.randint(150, 255)

        cv2.circle(
            forged,
            (x, y),
            radius,
            intensity,
            -1
        )

        cv2.circle(
            mask,
            (x, y),
            radius,
            255,
            -1
        )

    return forged, mask

In [7]:
def contrast_manipulation(img):

    forged = img.copy()

    alpha = random.uniform(1.3, 2.0)

    beta = random.randint(-20, 20)

    forged = cv2.convertScaleAbs(
        forged,
        alpha=alpha,
        beta=beta
    )

    mask = np.ones_like(img) * 255

    return forged, mask

In [9]:
def generate_forensic_dataset(dataset_root,
                               output_root):

    dataset_root = Path(dataset_root)

    class_to_paths = {}

    for cls in dataset_root.iterdir():

        if cls.is_dir():

            class_to_paths[cls.name] = list(
                cls.glob("*")
            )

    material_classes = [
        "Biological",
        "Fibres",
        "Nanowires",
        "Particles",
        "Porous_Sponge",
        "Powder"
    ]

    device_classes = [
        "Films_Coated_Surface",
        "MEMS_devices_and_electrodes",
        "Patterned_surface",
        "Tips"
    ]

    all_classes = list(class_to_paths.keys())

    for class_name in all_classes:

        print(f"\nProcessing {class_name}")

        image_paths = class_to_paths[class_name]

        for path in tqdm(image_paths):

            try:

                img = load_grayscale(path)

                filename = path.name

                # -----------------
                # Copy-Move
                # -----------------

                forged, mask = copy_move_forgery(img)

                save_output(
                    forged,
                    mask,
                    output_root,
                    "copy_move",
                    class_name,
                    filename
                )

                # -----------------
                # Synthetic Defects
                # -----------------

                forged, mask = synthetic_defect_injection(img)

                save_output(
                    forged,
                    mask,
                    output_root,
                    "synthetic_defect",
                    class_name,
                    filename
                )

                # -----------------
                # Contrast
                # -----------------

                forged, mask = contrast_manipulation(img)

                save_output(
                    forged,
                    mask,
                    output_root,
                    "contrast",
                    class_name,
                    filename
                )

                # -----------------
                # Cross-Domain Splicing
                # -----------------

                if class_name in material_classes:

                    target_class = random.choice(
                        device_classes
                    )

                else:

                    target_class = random.choice(
                        material_classes
                    )

                donor_path = random.choice(
                    class_to_paths[target_class]
                )

                donor_img = load_grayscale(
                    donor_path
                )

                donor_img = cv2.resize(
                    donor_img,
                    (img.shape[1],
                     img.shape[0])
                )

                forged, mask = splice_forgery(
                    img,
                    donor_img
                )

                save_output(
                    forged,
                    mask,
                    output_root,
                    "splicing",
                    class_name,
                    filename
                )

            except Exception as e:

                print(e)

In [11]:
generate_forensic_dataset(
    dataset_root="dataset/authentic",
    output_root="dataset/forensic"
)


Processing Patterned_surface


100%|███████████████████████████████████████| 3310/3310 [00:57<00:00, 58.02it/s]



Processing Biological


100%|█████████████████████████████████████████| 953/953 [00:20<00:00, 45.76it/s]



Processing Porous_Sponge


...

100%|███████████████████████████████████████| 3656/3656 [01:04<00:00, 56.68it/s]



Processing MEMS_devices_and_electrodes


100%|███████████████████████████████████████| 4158/4158 [01:11<00:00, 58.49it/s]



Processing Powder


100%|█████████████████████████████████████████| 895/895 [00:15<00:00, 56.42it/s]
